# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² human protein dataset using the `mlcroissant` library. All dataset entities are referenced by their `@id` per the Croissant standard.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description from the metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
List available record sets and their fields (using their `@id`).

In [ ]:
# The Croissant metadata allows access to record sets and their fields.
# Print each record set and its field/column info by @id.

for record_set in metadata.record_sets:
    print(f"Record Set ID: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields (by @id):")
    for field in record_set.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Always use the RecordSet and Field `@id` values from the above overview.

In [ ]:
# List all record set ids
record_set_ids = [r.id for r in metadata.record_sets]
dataframes = {}

# Load ALL record sets into DataFrames, mapped by their `@id`
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for RecordSet: {record_set_id}")
        print(f"Fields: {dataframes[record_set_id].columns.tolist()}")
        print(f"Preview:")
        display(dataframes[record_set_id].head())
        print()
    else:
        print(f"No records found for RecordSet: {record_set_id}")

# For demonstration, select the first available record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Using record set '{main_record_set_id}' for analysis.")
else:
    print("No record set data available for analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing: filter records, normalize fields, group by attributes. All field names used here should be listed in the RecordSet fields above (`@id`).

In [ ]:
import numpy as np
# Identify a numeric field (`@id`) from the overview. In this dataset, likely candidates are MW, CoveragePct, Peptides, etc.
# Please replace 'MW' and 'Group' (example group field) with actual field @ids as per the record set overview printed above.

numeric_field_id = None
group_field_id = None

# Try to auto-select numeric field and group field for demo (e.g. MW, CoveragePct, etc.)
if main_record_set_id is not None:
    cols = dataframes[main_record_set_id].columns
    # Priority order for numeric field
    for candidate in ['cr:MW', 'cr:CoveragePct', 'cr:Peptides', 'cr:UniquePeptides', 'cr:Abundance']:
        if candidate in cols:
            numeric_field_id = candidate
            break
    # For grouping, try 'cr:Sample', 'cr:Description', etc.
    for candidate in ['cr:Sample', 'cr:Description', 'cr:Gene', 'cr:Accession']:
        if candidate in cols:
            group_field_id = candidate
            break

# Fallback: pick the first numeric-looking column
if numeric_field_id is None and main_record_set_id is not None:
    for col in dataframes[main_record_set_id].columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][col]):
            numeric_field_id = col
            break

if main_record_set_id is not None and numeric_field_id is not None:
    df = dataframes[main_record_set_id]
    print(f"Processing numeric field (by @id): {numeric_field_id}")
    # Filter for values > 10
    threshold = 10
    # Coerce non-numeric to NaN to perform filtering
    df_num = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df_num > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df[[numeric_field_id]].head())

    # Normalize field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
    ) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        # Only average numeric columns
        grouped_df = filtered_df.groupby(group_field_id, as_index=False).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean of numeric fields):")
        display(grouped_df.head())
else:
    print("Unable to identify a numeric field in the chosen record set.")

## 5. Visualization
Visualize distributions or relationships between fields. Example: Histogram of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field_id, y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load a FAIR² Croissant dataset, review its schema (by `@id`), extract record sets and fields, and perform basic EDA using the `mlcroissant` library.

- **Key observations**: The dataset provides mass spectrometry-derived protein parameters across samples. You can further explore by referencing the schema's `@id` fields and extending the processing.

**Next steps**: Apply more advanced analytics, integrate with other FAIR datasets, or use the robust Croissant metadata to automate data analysis across similar datasets!